In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

# 1. Navegación dinámica: Buscamos el archivo pyproject.toml hacia arriba
def find_project_root(current_path, target="pyproject.toml"):
    for parent in Path(current_path).parents:
        if (parent / target).exists():
            return parent
    return None

root = find_project_root(os.getcwd())

if root:
    print(f"📂 Proyecto detectado en: {root}")
    # 2. Instalación en modo editable (-e)
    # El flag --no-deps es opcional si solo quieres registrar los cambios de archivos
    !pip install -e "{root}"
    
    print("\n✅ Instalación completada. Ya puedes importar 'legion_goes' desde cualquier celda.")
else:
    print("❌ Error: No se encontró 'pyproject.toml'. Asegúrate de estar dentro de la estructura de MAIE_tesis_github.")

📂 Proyecto detectado en: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
Obtaining file:///home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for legion-goes (pyproject.toml) ... done
  Created wheel for legion-goes: filename=legion_goes-0.1.9-py3-none-any.whl size=2701 sha256=2cb573997f20247a21bd2324784bd71e814cbb515bee1ecb0bf2c723115accb0
  Stored in directory: /tmp/pip-ephem-wheel-cache-fsksyyjj/wheels/45/f6/84/0a48d0659fd5307d2efb8dfac8c3238f417bd9a968b770cecf
Successfully built legion-goes
  Attempting uninstall: legion-goes
    Found existing installation: legion-goes 0.1.9
    Uninstalling legion-goes-0.1.9:
      Successfully uninstalled legion-goes-0.1.9

✅ Instalación completada. Ya puedes 

In [3]:
try:
    import legion_goes
    print(f"📦 Librería 'legion_goes' lista para usar.")
except ImportError:
    print("⚠️ Instalación terminada, pero puede que necesites reiniciar el Kernel.")

📦 Librería 'legion_goes' lista para usar.


In [5]:
# Imports limpios desde la librería instalada
from legion_goes.tasks.task02_download.actions.action01_gen_plan_download import run_task02_download_action01_generate_plan
from legion_goes.tasks.task02_download.actions.action02_check_plan_download import execute_task02_download_action02_check_plan
from legion_goes.tasks.task02_download.actions.action03_run_plan_download import execute_task02_download_action03_run_download
from legion_goes.tasks.task02_download.actions.fn01_file_name_plan_download import generate_plan_download_file_path
from pathlib import Path


In [6]:
# 1. Configuración
cfg = {
    "sat_id": "19",
    "year": "2026",
    "day": "065",#"003",
    "product_id": "ABI-L2-MCMIPF",
    "output_folder_base": Path("./data/goes_test").resolve()
}

# 2. Generar el Plan (Action 01)
run_task02_download_action01_generate_plan(**cfg)

# 3. Sincronizar Disco (Action 02)
execute_task02_download_action02_check_plan(**cfg)

# 4. Descarga Masiva (Action 03)
# Recuperamos la ruta del JSON para pasársela al Downloader
path_plan = generate_plan_download_file_path(**cfg)

execute_task02_download_action03_run_download(
    path_plan=path_plan, 
    threads=4, 
    checkpoint_n=10
)


🚀 [Action01 - Generator Plan Download]
⚠️  Status: Plan exists. Skipping.
📂 Path: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/tests/test_tasks/test_task02_download/test_actions/data/goes_test/2026/065/plan_01-download_2026_065_GOES19_EAST_ABI-L2-MCMIPF.json


🔍 [SCAN] Checking local integrity: ABI-L2-MCMIPF | 2026065 (GOES19 - EAST)
    ... scanned 100/144 items
    ... scanned 144/144 items

 ✅ Scan finished: 61/144 files found.

🔍 [PRE-CHECK] Synchronizing local inventory...
    ... scanned 100/144 items
    ... scanned 144/144 items
🧹 [CLEANUP] Scanning for temporary files...
🔍 [SCANNING] Fetching full day inventory from S3 bucket: noaa-goes19...

═══════════════════════════════════════════════════════════════════════════════════════════════
 🕒 SYSTEM TIME: 2026-03-06 11:18:31 | UTC: 10:18:31
 📅 DAILY DOWNLOAD MONITOR (Legion Goes v.0.0.1)
═══════════════════════════════════════════════════════════════════════════════════════════════
  HOUR  ║    EXP     │     S3   

True

In [7]:
from legion_goes.tasks.task03_proc_single.actions.action01_gen_plan_proc_single import run_action01_gen_plan_proc_single
from legion_goes.tasks.task03_proc_single.actions.action02_check_plan_proc_single import run_action02_check_plan_proc_single
from legion_goes.tasks.task03_proc_single.actions.action03_run_plan_proc_single import run_action03_run_plan_proc_single

# Configuración del Job
job = {
    "year": "2026", "day": "062", "sat_id": "19", 
    "product_id": "ABI-L2-MCMIPF", "fnp_tag": "fnp01", # <--- fnp01 o fnp02
    "path_download_base":  Path("./data_proc").resolve()
}

# 1. Crear el Plan (Genera el JSON)
path_p = run_action01_gen_plan_proc_single(**job)

# 2. Auditar (Verifica qué falta)
run_action02_check_plan_proc_single(path_p)

# 3. ¡PROCESAR!
run_action03_run_plan_proc_single(path_p, overwrite=False)

FileNotFoundError: [Action01 - Gen Plan Proc] No existe el plan de descarga en: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/tests/test_tasks/test_task02_download/test_actions/data_proc/2026/062/plan_01-download_2026_062_GOES19_EAST_ABI-L2-MCMIPF.json

In [ ]:
# =============================================================================
# FILE PATH: src/legion_goes/sot/goes_sat.py
# Version: 1.0.4 (Object-Driven Resolution)
# =============================================================================

from types import MappingProxyType
from datetime import datetime

# ===================================================================
# CONFIGURATION & HISTORICAL TRANSITIONS
# ===================================================================
AVAILABLE_GOES_SAT_POSITIONS = ("east", "west")

# Exact operational transition dates (Source: NOAA)
TRANSITIONS = {
    "east_16_to_19": datetime(2025, 2, 10), 
    "west_17_to_18": datetime(2023, 1, 4),  
    "east_13_to_16": datetime(2017, 12, 18) 
}

_PRIVATE_SAT_INFO = {
    "16": {"id": "16", "bucket": "noaa-goes16", "name01": "16", "name02": "G16", "name03": "GOES16", "position": "east"},
    "17": {"id": "17", "bucket": "noaa-goes17", "name01": "17", "name02": "G17", "name03": "GOES17", "position": "west"},
    "18": {"id": "18", "bucket": "noaa-goes18", "name01": "18", "name02": "G18", "name03": "GOES18", "position": "west"},
    "19": {"id": "19", "bucket": "noaa-goes19", "name01": "19", "name02": "G19", "name03": "GOES19", "position": "east"}
}

AVAILABLE_GOES_ID = tuple(_PRIVATE_SAT_INFO.keys())
# ===================================================================
# IMMUTABILITY ENGINE
# ===================================================================
def _make_deep_immutable(obj):
    if isinstance(obj, dict):
        return MappingProxyType({k: _make_deep_immutable(v) for k, v in obj.items()})
    elif isinstance(obj, (list, tuple)):
        return tuple(_make_deep_immutable(i) for i in obj)
    return obj

SAVED_INFO_SAT_GOES = _make_deep_immutable(_PRIVATE_SAT_INFO)

# ===================================================================
# PUBLIC INTERFACE (Object-Driven)
# ===================================================================

def get_SOT_sat_id_from_date_and_position(position: str, year: str = None, day: str = None) -> str:
    """
    Determines which satellite ID was active in a position at a given date.
    It uses the logic of TRANSITIONS to pick from SAVED_INFO_SAT_GOES.
    """
    # 1. Date Handling
    if not year or not day:
        now = datetime.now()
        year, day = now.strftime("%Y"), now.strftime("%j")
    
    try:
        date_obj = datetime.strptime(f"{year}-{str(day).zfill(3)}", "%Y-%j")
    except ValueError:
        raise ValueError(f"Invalid date: Year {year}, Day {day}")
        
    pos = str(position).lower().strip()

    # 2. Historical Routing (Resolves to ID)
    if pos == "east":
        return "19" if date_obj >= TRANSITIONS["east_16_to_19"] else "16"
            
    if pos == "west":
        return "18" if date_obj >= TRANSITIONS["west_17_to_18"] else "17"
            
    raise ValueError(f"Unknown position: {position}. Available: {AVAILABLE_GOES_SAT_POSITIONS}")

######################################################################################################################
def get_SOT_position_from_sat_id(sat_id: str) -> str:
    """
    Returns the physical position ('east' or 'west') of a specific satellite ID.
    Now it resolves using the object's default position.
    """
    # 1. Clean and Validate ID
    val_id = str(sat_id).lower().replace("goes", "").replace("-", "").strip()
    
    if val_id not in SAVED_INFO_SAT_GOES:
        raise ValueError(f"Unknown Satellite ID: {sat_id}")

    # 2. Object-Driven Resolution
    # We return the 'position' property stored in our immutable Master Object
    return SAVED_INFO_SAT_GOES[val_id]["position"]

######################################################################################################################
def get_SOT_goes_info_sat(sat_id: str) -> MappingProxyType:
    """
    Universal Resolver: Returns the full metadata object.
    Automatically detects if input is a position or a direct ID.
    """
    val = str(sat_id).lower().replace("goes", "").replace("-", "").strip()

    # If it's a position (east/west), resolve to ID first
    if val in AVAILABLE_GOES_SAT_POSITIONS:
        real_id = get_SOT_sat_id_from_date_and_position(val, year, day)
    else:
        # If it's an ID (16, 17...), use it directly
        real_id = val

    if real_id not in SAVED_INFO_SAT_GOES:
        raise ValueError(f"Could not resolve Satellite Info for: {sat_id_or_pos}")

    return SAVED_INFO_SAT_GOES[real_id]

# ===================================================================
# UNIT TESTING
# ===================================================================
if __name__ == "__main__":
    print("\n" + " TEST: OBJECT-DRIVEN SATELLITE RESOLUTION ".center(60, "="))
    
    # Test 1: Position -> ID (Historical)
    id_2024 = get_SOT_sat_id_from_date_and_position(position="east", year="2024", day="100")
    print(f"✅ East in 2024: {id_2024} (Expected: 16)")

    # Test 2: ID -> Position (From Object)
    pos_check = get_SOT_position_from_sat_id(sat_id="19")
    print(f"✅ Position of Sat 19: {pos_check} (Expected: east)")

    # Test 3: Get Full Info Object from Position
    info = get_SOT_goes_info_sat(sat_id = "19")
    print(f"✅ Info via Position (West 2022): {info['name03']} | Bucket: {info['bucket']}")

    print("="*60 + "\n")

In [ ]:
from pathlib import Path

# --- PARÁMETROS DE ENTRADA ---
SAT_ID = "19"
YEAR = 2026
DAY = 3
PRODUCT_ID = "ABI-L2-MCMIPF"
OUTPUT_FOLDER = Path("./data/goes_test").resolve() # Ruta absoluta para evitar líos
THREADS = 8           # Ajusta según la potencia de tu conexión en Legion
CHECKPOINT_EVERY = 5  # Guarda el JSON cada 5 archivos descargados

In [ ]:
from pathlib import Path

# 1. Define tu base (el punto actual o una ruta absoluta en Legion)
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"
# 2. Ejecución corregida
success = run_action01_generate_plan(
    sat_id="19",
    year="2026",
    day="003",
    product_id="ABI-L2-MCMIPF",
    # Corregido: Nombre del argumento y construcción de la ruta
    output_folder_base = output_folder_base, 
    overwrite=True
)

if success:
    print("🚀 PROCESO COMPLETADO: El JSON ha sido creado con la estructura v.1.1.0.")

In [ ]:
# =============================================================================
# JUPYTER TEST: Action 02 - Check Plan Integrity
# =============================================================================
from pathlib import Path
from legion_goes.tasks.task02_download.actions.action02_check_plan_download import execute_action_check_plan

# 1. Definir EXACTAMENTE la misma ruta que usaste para crear el plan
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"

# 2. Ejecución del Checker (Action 02)
# Nota: sat_id debe coincidir con el del Plan ("19")
success_check = execute_action_check_plan(
    sat_id="19",
    year=2026,
    day=3,
    product_id="ABI-L2-MCMIPF",
    output_folder_base=str(output_folder_base)
)

if success_check:
    print("\n✅ CHECKER COMPLETADO: El inventario local ha sido actualizado en el JSON.")
else:
    print("\n❌ ERROR: No se pudo realizar el chequeo. Verifica que el archivo .json existe.")

In [ ]:
from legion_goes.tasks.task02_download.actions.action03_run_plan_download import execute_action_run_download

# Usamos la misma ruta de tus pruebas anteriores
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"

# ¡A descargar!
success_dl = execute_action_run_download(
    sat_id="19",
    year=2026,
    day=3,
    product_id="ABI-L2-MCMIPF",
    output_folder_base=str(output_folder_base),
    threads=4,       # <--- 4 a 8 hilos es lo ideal para no saturar Legion
    overwrite=False  # Solo baja lo que el Checker marcó como faltante
)